# Clase 03 — Preprocesamiento lingüístico

**Curso:** NLP Clásico con Python · **Bloque 1:** Del texto a los tokens
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/03_preprocesamiento_linguistico.md`

## Qué vas a ver en esta clase

- **Stopwords con criterio**: qué contienen las listas de NLTK en español y en italiano (¡también negaciones!), cuándo ayudan y cuándo cambian el significado de una frase.
- **Stemming** con Snowball: recortar por reglas, con sobre-stemming y sub-stemming.
- **Lematización** con spaCy: el lema depende de la categoría gramatical (POS, aquí solo una introducción mínima; se estudia a fondo en la clase 7) y los modelos pequeños se equivocan de formas previsibles.
- **Comparación NLTK frente a spaCy** en español y en italiano.
- **Un experimento guiado**: cómo cambia el tamaño del vocabulario (tipos) tras cada paso, en reseñas y tickets, en los dos idiomas.
- Una **guía de decisión**: qué preprocesamiento para qué tarea.

## Requisitos

- La clase 1 (qué es el NLP) y la clase 2 (limpieza y tokenización): aquí tokenizamos con `re.findall(r"\w+", ...)`.
- Librerías: `nltk`, `spacy` (con `es_core_news_sm` e `it_core_news_sm`), `pandas`, `matplotlib`. La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso, junto al notebook o en `/content/datos/` (en Colab, súbela desde el panel de archivos).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Los corpus son sintéticos y pequeños (*ejemplo ilustrativo*): las cifras enseñan un patrón, no son generalizables. Las salidas de spaCy dependen de la versión (aquí, spaCy 3.8 y modelos 3.8.0) y pueden variar ligeramente con otras.

---
## Instalación y recursos

Esta celda instala lo que falte, comprueba las listas de stopwords de NLTK y carga los dos modelos de spaCy. Si ya lo tienes todo, no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("nltk", "nltk")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import nltk
import spacy

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

def cargar_modelo(nombre):
    """Carga un modelo de spaCy y lo descarga si no está instalado."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Recursos de NLTK y modelos de spaCy listos.")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla** y definimos tres cosas que usaremos todo el rato: `cargar_datos` (busca la carpeta `datos/`), `tokenizar` (la de la clase 2) y los recursos por idioma: listas de stopwords (`SW`), stemmers Snowball (`STEMMER`) y modelos de spaCy (`NLP`). Los idiomas se indican con `"spa"` (español) e `"ita"` (italiano).

In [ ]:
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

from pathlib import Path

def cargar_datos(nombre):
    """Busca un archivo en datos/, ../datos/ y /content/datos/ y devuelve su ruta."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = Path(base) / nombre
        if ruta.exists():
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        f"No encuentro {nombre}. Sube la carpeta datos/ (en Colab: panel de archivos) o ejecuta el notebook desde su carpeta.")


def tokenizar(texto):
    """Tokenización de la clase 2: palabras con re.findall."""
    return re.findall(r"\w+", texto)

NOMBRE_NLTK = {"spa": "spanish", "ita": "italian"}
SW = {idioma: set(stopwords.words(nombre)) for idioma, nombre in NOMBRE_NLTK.items()}
STEMMER = {idioma: SnowballStemmer(nombre) for idioma, nombre in NOMBRE_NLTK.items()}
NLP = {"spa": nlp_es, "ita": nlp_it}
print("Setup listo.")

---
## 1. Stopwords con criterio

Una **stopword** es una palabra muy frecuente y de poco contenido (`el`, `de`, `che`). Quitarlas reduce el ruido cuando cuentas palabras. Pero una lista de stopwords es una decisión, no una verdad: veamos qué hay dentro.

In [ ]:
for idioma in ("spa", "ita"):
    lista = sorted(SW[idioma])
    print(idioma, "→", len(lista), "palabras; primeras 12:", lista[:12])

In [ ]:
# datos-es
FRASES_NEGACION = {
    "spa": ["no está bien", "la entrega no fue rápida", "sin problemas", "no llegó nada"],
    "ita": ["non è buono", "la consegna non è stata veloce", "senza problemi", "non è arrivato niente"],
}
NEGACIONES = {
    "spa": {"no", "nunca", "jamás", "ni", "sin", "nada", "nadie", "tampoco"},
    "ita": {"non", "mai", "né", "senza", "niente", "nessuno", "nulla", "neanche"},
}
DOMINIO = {"spa": {"pedido", "pedidos"}, "ita": {"ordine", "ordini"}}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for palabra in sorted(NEGACIONES[idioma]):
        filas.append({"idioma": idioma, "palabra": palabra, "¿está en la lista de NLTK?": palabra in SW[idioma]})
tabla_neg = pd.DataFrame(filas)
print(tabla_neg.to_string(index=False))

**¿Qué observas?** Las listas contienen algunas negaciones (`no`, `ni`, `sin`, `nada` en español; `non` en italiano) pero **no todas**: `nunca`, `jamás`, `nadie` y `tampoco` no están en la lista española, y en italiano solo `non`. Las dos listas no son equivalentes: quitar stopwords en español elimina `sin`, y en italiano `senza` se queda. Con `spaCy` o con cualquier otra lista el reparto sería distinto: **siempre hay que mirar la lista**.

### 1.1 Cuando quitar stopwords cambia el significado

Aplicamos el filtro estándar (`quitar_stopwords`) a cuatro frases en cada idioma. Compara lo que queda con lo que decía la frase.

In [ ]:
def quitar_stopwords(tokens, idioma):
    """Filtro estándar: quita todo lo que esté en la lista de NLTK."""
    return [t for t in tokens if t.lower() not in SW[idioma]]

for idioma in ("spa", "ita"):
    for frase in FRASES_NEGACION[idioma]:
        print(f"{frase:36} → {quitar_stopwords(tokenizar(frase), idioma)}")

**¿Qué observas?** `no está bien` se queda en `['bien']`: un texto que dice lo contrario del original. `la entrega no fue rápida` queda como `['entrega', 'rápida']` (¡la palabra `fue` también es stopword!) y `sin problemas` (positivo) queda como `['problemas']`, que parece una queja. En italiano, `non è buono` se queda en `['buono']`, pero `senza problemi` conserva `senza`. Para una bolsa de palabras que solo cuenta, `bien` y `problemas` son las señales que el clasificador verá: la negación ha desaparecido.

### 1.2 Una lista personalizada

La solución no es «no usar stopwords», sino **construir la lista con criterio**: partir de la de NLTK, **conservar las negaciones** y, si el problema lo pide, **añadir palabras de dominio** (en un buscador de tickets, `pedido`/`ordine` aparecen en casi todos y aportan poco para distinguirlos).

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def quitar_stopwords_pers(tokens, idioma):
    return [t for t in tokens if t.lower() not in SW_PERS[idioma]]

for idioma in ("spa", "ita"):
    print(idioma, "NLTK:", len(SW[idioma]), "→ personalizada:", len(SW_PERS[idioma]))
    for frase in FRASES_NEGACION[idioma]:
        print(f"   {frase:36} → {quitar_stopwords_pers(tokenizar(frase), idioma)}")

In [ ]:
# Reseñas: cuántos tokens son stopwords y cuántas reseñas contienen una negación de la lista de NLTK
resenas = {"spa": pd.read_csv(cargar_datos("resenas_es.csv")), "ita": pd.read_csv(cargar_datos("resenas_it.csv"))}
tickets = {"spa": pd.read_csv(cargar_datos("tickets_es.csv")), "ita": pd.read_csv(cargar_datos("tickets_it.csv"))}

filas = []
for idioma in ("spa", "ita"):
    minus = [[t.lower() for t in tokenizar(x)] for x in resenas[idioma]["texto"]]
    n_tokens = sum(len(l) for l in minus)
    n_stop = sum(t in SW[idioma] for l in minus for t in l)
    negaciones_lista = NEGACIONES[idioma] & SW[idioma]
    con_neg = sum(any(t in negaciones_lista for t in l) for l in minus)
    filas.append({"idioma": idioma, "tokens": n_tokens, "% stopwords": round(100 * n_stop / n_tokens, 1),
                  "reseñas con negación de la lista": con_neg, "de": len(minus)})
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Cerca de la mitad de los tokens son stopwords (así se ahorra memoria y ruido), pero una parte importante de las reseñas contiene una negación de la lista: son justo las reseñas en las que el filtro estándar puede invertir el sentido. Recuerda que el corpus es sintético y pequeño (*ejemplo ilustrativo*): mira el patrón, no la cifra exacta.

---
## 2. Stemming con Snowball

El **stemming** recorta cada palabra hasta una **raíz** aplicando reglas de sufijos. No usa diccionario, así que **no garantiza obtener una palabra real** (`universidad` → `univers`). Es rápido y determinista. `SnowballStemmer("spanish")` y `SnowballStemmer("italian")` son los de NLTK.

In [ ]:
# datos-es
PALABRAS = {
    "spa": ["universidad", "universo", "generar", "generación", "generoso", "general",
            "organizar", "órgano", "viene", "venir", "fue", "fueron", "ir", "iba", "voy", "bueno", "mejor"],
    "ita": ["università", "universo", "generare", "generazione", "generoso", "generale",
            "organizzare", "organo", "viene", "venire", "fu", "furono", "andare", "andava", "vado", "buono", "migliore"],
}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for palabra in PALABRAS[idioma]:
        filas.append({"idioma": idioma, "forma": palabra, "raíz Snowball": STEMMER[idioma].stem(palabra)})
tabla_raices = pd.DataFrame(filas)
print(tabla_raices.to_string(index=False))

### 2.1 Sobre-stemming y sub-stemming

Dos errores típicos, que se ven agrupando las palabras por su raíz:

- **Sobre-stemming** (*over-stemming*): palabras **distintas** con la **misma** raíz.
- **Sub-stemming** (*under-stemming*): formas de la **misma** palabra con raíces **distintas** (sobre todo en verbos irregulares).

In [ ]:
for idioma in ("spa", "ita"):
    grupos = {}
    for palabra in PALABRAS[idioma]:
        grupos.setdefault(STEMMER[idioma].stem(palabra), []).append(palabra)
    print(idioma)
    for raiz, formas in grupos.items():
        if len(formas) > 1:
            print(f"   misma raíz {raiz!r:12} ← {formas}")
    print("   raíces distintas para formas de venir/ir (sub-stemming):",
          sorted({STEMMER[idioma].stem(p) for p in PALABRAS[idioma][8:16] if p not in ("bueno", "buono")}))

In [ ]:
# Un stemmer de otro idioma estropea el texto: el stemmer inglés sobre palabras españolas e italianas
STEMMER_EN = SnowballStemmer("english")
filas = []
for idioma, palabras in (("spa", ["cantaremos", "universidad", "organización", "entregado"]), ("ita", ["canteremo", "università", "organizzazione", "consegnato"])):  # dato-es
    for palabra in palabras:
        filas.append({"idioma": idioma, "forma": palabra, "Snowball del idioma": STEMMER[idioma].stem(palabra), "Snowball inglés": STEMMER_EN.stem(palabra)})
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** `universidad` y `universo` comparten raíz (`univers`), igual que `generar`, `generación` y `generoso` (`gener`): sobre-stemming. En cambio `viene`, `venir`, `fue`, `iba`, `voy` o `andare`, `vado`, `andava` tienen raíces distintas: sub-stemming. Y el stemmer inglés apenas toca las palabras españolas o italianas (o las recorta mal): usar el stemmer equivocado no da error, solo un resultado peor.

---
## 3. Lematización con spaCy

La **lematización** devuelve el **lema**: la forma de diccionario (`cantaron` → `cantar`, `ordini` → `ordine`). Usa un modelo estadístico y reglas, y **depende de la categoría gramatical** (POS, *part of speech*). Aquí solo necesitas cuatro etiquetas universales (UPOS): `NOUN` (sustantivo), `VERB` (verbo), `ADJ` (adjetivo) y `AUX` (auxiliar); la clase 7 las estudia a fondo.

### 3.1 El mismo token, dos lemas

In [ ]:
# datos-es
CASOS_POS = {
    "spa": ["Ayer vino mi hermana", "Con la cena tomamos vino tinto", "Yo trabajo mucho", "El trabajo es difícil"],
    "ita": ["Io lavoro molto", "Il lavoro è difficile", "Il vino è buono", "Ho pianto tutta la notte"],
}

In [ ]:
for idioma in ("spa", "ita"):
    for frase in CASOS_POS[idioma]:
        doc = NLP[idioma](frase)
        print(f"{frase:34} →", [(t.text, t.lemma_, t.pos_) for t in doc])

**¿Qué observas?** En español, `vino` es el verbo *venir* en `Ayer vino mi hermana` (lema `venir`) y la bebida en `tomamos vino tinto` (lema `vino`); `trabajo` es el verbo `trabajar` o el sustantivo `trabajo` según la frase. En italiano pasa igual con `lavoro`. Un lematizador que ve solo la forma aislada no puede decidir: necesita la frase. Fíjate también en un error del modelo pequeño: en `Il vino è buono`, `buono` puede salir etiquetado como `VERB` (con spaCy 3.8; puede variar según la versión): la etiqueta se equivoca, aunque el lema sea correcto. **No te fíes del POS sin mirarlo.**

### 3.2 Errores típicos del modelo pequeño

Los modelos `_sm` son rápidos y pequeños, y fallan en cuatro situaciones: **clíticos** (`dámelo`, `dammelo`, ya vistos en la clase 1), **formas aisladas** (una palabra sola no da contexto), **nombres propios** (mayúsculas al principio de frase) y **texto ruidoso**. Con spaCy 3.8 y `es_core_news_sm`/`it_core_news_sm` 3.8.0 (pueden variar ligeramente con otras versiones):

In [ ]:
# datos-es
CASOS_ERROR = {
    "spa": ["Dámelo mañana", "dame el paquete", "Quiero devolverlo hoy", "Miguel de Cervantes vive en Madrid", "vino", "Hablo con Ana"],
    "ita": ["Dammelo domani", "Passamelo domani", "Vorrei restituirlo", "Marco Rossi vive a Milano", "vino", "Io canto ogni sera"],
}

In [ ]:
for idioma in ("spa", "ita"):
    for frase in CASOS_ERROR[idioma]:
        doc = NLP[idioma](frase)
        print(f"{frase:36} →", [(t.text, t.lemma_, t.pos_) for t in doc])

**¿Qué observas?** Los clíticos dan lemas raros: `Dámelo` se convierte en `dámelir`, `devolverlo` en `devolver él` y `Dammelo` se queda como nombre propio sin lematizar; `Hablo` al comienzo de frase se toma por nombre propio; una palabra suelta como `vino` recibe el lema que el modelo considera más probable, sin contexto (en español, `venir`). No hay error visible: el lema simplemente es dudoso. Por eso conviene **revisar una muestra** de lo que sale antes de fiarse.

### 3.3 Texto ruidoso: `texto` frente a `texto_limpio`

Las reseñas del corpus tienen una versión bruta (`texto`, con mayúsculas, acentos perdidos, alargamientos, abreviaturas de chat) y una limpia (`texto_limpio`). Comparamos los lemas de las mismas reseñas.

In [ ]:
def lemas_de(texto, idioma):
    """Lemas en minúscula de los tokens alfanuméricos de un texto (contexto = el texto entero)."""
    return [t.lemma_.lower() for t in NLP[idioma](texto) if re.fullmatch(r"\w+", t.text)]

for idioma in ("spa", "ita"):
    df = resenas[idioma]
    ruidosas = df[(df["texto"] != df["texto_limpio"]) & (df["texto"].str.len() < 95)].head(2)
    for _, fila in ruidosas.iterrows():
        print(idioma, fila["id"])
        print("   bruto: ", fila["texto"])
        print("           →", lemas_de(fila["texto"], idioma))
        print("   limpio:", fila["texto_limpio"])
        print("           →", lemas_de(fila["texto_limpio"], idioma))

**¿Qué observas?** Los alargamientos (`reservaaa`) y las erratas (`cacnelar`, `lascinao`) pasan tal cual a la lista de lemas o se lematizan mal: la misma reseña da listas distintas. La limpieza de la clase 2 **antes** de lematizar no es opcional: el lematizador es tan bueno como el texto que recibe.

---
## 4. NLTK frente a spaCy: una tabla

Para cada forma usamos una frase corta como contexto (spaCy la necesita) y mostramos la raíz Snowball de la forma y el lema y el POS que da spaCy **para ese token**. Las cifras de spaCy dependen de la versión.

In [ ]:
# datos-es
COMPARACION = {
    "spa": [("cantaron", "Ellos cantaron muy bien"), ("mejores", "Los mejores productos"), ("pedidos", "Los pedidos llegaron tarde"),
            ("entregado", "Me han entregado el paquete"), ("fueron", "Los paquetes fueron entregados"), ("vino", "Ayer vino mi hermana"),
            ("dámelo", "Dámelo mañana"), ("universidad", "La universidad y el universo"), ("Cervantes", "Miguel de Cervantes vive en Madrid")],
    "ita": [("cantarono", "Loro cantarono molto bene"), ("migliori", "I migliori prodotti"), ("ordini", "Gli ordini sono arrivati in ritardo"),
            ("consegnato", "Mi hanno consegnato il pacco"), ("furono", "I pacchi furono consegnati"), ("vino", "Il vino è buono"),
            ("dammelo", "Dammelo domani"), ("università", "L'università e l'universo"), ("Rossi", "Marco Rossi vive a Milano")],
}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for forma, frase in COMPARACION[idioma]:
        doc = NLP[idioma](frase)
        token = next(t for t in doc if t.text.lower() == forma.lower())
        filas.append({"idioma": idioma, "forma": forma, "raíz Snowball": STEMMER[idioma].stem(forma.lower()),
                      "lema spaCy": token.lemma_, "POS": token.pos_})
tabla_comp = pd.DataFrame(filas)
print(tabla_comp.to_string(index=False))

**¿Qué observas?** Compara columna a columna. La raíz nunca es una palabra «bonita» (`ped`, `entreg`), pero agrupa formas relacionadas; el lema es una palabra real cuando el modelo acierta, y falla en clíticos (`dámelo` → `dámelir`) y en formas poco frecuentes (`cantarono` no se lematiza). El stemmer, por su parte, recorta también los nombres propios (`Cervantes` → `cervant`, `Rossi` → `ross`), que el lematizador deja intactos. Y las formas irregulares (`fueron`, `furono`) se lematizan bien con spaCy pero se quedan sueltas en el stemmer.

---
## 5. Experimento guiado: el efecto en el vocabulario

**Pregunta:** ¿cuánto se reduce el vocabulario en cada paso? Medimos el número de **tipos** (palabras distintas) y el cociente **tipos/tokens** sobre las reseñas y los tickets, en los dos idiomas, tras cada paso: bruto → minúsculas → sin stopwords → stemming → lematización. Ojo: **no medimos la calidad de un clasificador** (eso, en la clase 5), solo el tamaño del vocabulario. Tokenizamos con `re.findall` como en la clase 2. Stemming y lematización parten de los tokens sin stopwords (son dos alternativas, no una detrás de otra).

In [ ]:
PASOS = ["bruto", "minúsculas", "sin stopwords", "stemming", "lematización"]

def medir(textos, idioma):
    """Devuelve {paso: lista de tokens} para un conjunto de textos."""
    brutos = [t for texto in textos for t in tokenizar(texto)]
    minusculas = [t.lower() for t in brutos]
    sin_sw = [t for t in minusculas if t not in SW[idioma]]
    raices = [STEMMER[idioma].stem(t) for t in sin_sw]
    lemas = []
    for doc in NLP[idioma].pipe(list(textos), disable=["ner", "parser"]):
        for t in doc:
            if re.fullmatch(r"\w+", t.text) and t.text.lower() not in SW[idioma]:
                lemas.append(t.lemma_.lower())
    return dict(zip(PASOS, [brutos, minusculas, sin_sw, raices, lemas]))

def resumen(pasos):
    filas = []
    for paso, lista in pasos.items():
        tipos = len(set(lista))
        filas.append({"paso": paso, "tokens": len(lista), "tipos": tipos, "tipos/tokens": round(tipos / len(lista), 3)})
    return pd.DataFrame(filas)

corpus = {
    ("reseñas", "spa"): resenas["spa"]["texto"], ("reseñas", "ita"): resenas["ita"]["texto"],
    ("tickets", "spa"): tickets["spa"]["texto"], ("tickets", "ita"): tickets["ita"]["texto"],
}
medidas = {clave: medir(textos, clave[1]) for clave, textos in corpus.items()}
for (nombre, idioma), pasos in medidas.items():
    print(nombre, idioma)
    print(resumen(pasos).to_string(index=False))
    print()

In [ ]:
tipos = pd.DataFrame({f"{nombre} ({idioma})": [len(set(l)) for l in pasos.values()] for (nombre, idioma), pasos in medidas.items()}, index=PASOS)
porcentaje = 100 * tipos / tipos.iloc[0]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
tipos.plot(kind="bar", ax=ejes[0], rot=25)
ejes[0].set_ylabel("tipos (palabras distintas)")
ejes[0].set_title("Vocabulario tras cada paso")
ejes[0].legend(fontsize=8)
porcentaje.plot(marker="o", ax=ejes[1], rot=25)
ejes[1].set_ylabel("% del vocabulario bruto")
ejes[1].set_title("Vocabulario relativo al bruto")
ejes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(porcentaje.round(1).to_string())

**¿Qué observas?** Tres cosas. **(1)** Pasar a minúsculas ya reduce el vocabulario entre un 11 % y un 14 %: `Pedido` y `pedido` dejan de ser dos tipos. **(2)** Quitar stopwords casi no cambia el número de tipos (solo desaparecen unos 90-110 tipos, aunque son los tokens más frecuentes, cerca de la mitad del texto). Por eso el cociente tipos/tokens **sube**: hay menos tokens, no más variedad. **(3)** El stemming reduce el vocabulario más que la lematización, porque agrupa más (incluidas palabras que no debería agrupar: sobre-stemming). ¿Cuánta reducción es útil? La que une formas de la misma palabra; la que une palabras distintas es información destruida. Las cifras son de corpus sintéticos (*ejemplo ilustrativo*).

### 5.1 ¿Y con el texto limpio?

Repetimos la medición de lemas sobre `texto_limpio` (reseñas): si el vocabulario de lemas baja mucho, parte de lo que parecía variedad de palabras era ruido.

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    bruto = medir(resenas[idioma]["texto"], idioma)
    limpio = medir(resenas[idioma]["texto_limpio"], idioma)
    for paso in PASOS:
        filas.append({"idioma": idioma, "paso": paso, "tipos (bruto)": len(set(bruto[paso])), "tipos (limpio)": len(set(limpio[paso]))})
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** El texto limpio tiene menos tipos en todos los pasos: el ruido (mayúsculas, erratas, alargamientos) infla el vocabulario y reduce la utilidad de la lematización. Limpiar primero (clase 2) y normalizar después (esta clase) es el orden natural.

---
## 6. Guía de decisión: qué preprocesamiento para qué tarea

No hay un preprocesamiento universal. Esta tabla resume una guía razonable; se comprueba con datos, no con fe.

In [ ]:
guia = pd.DataFrame([
    ("Búsqueda (clase 11)", "opcional; con criterio", "útil (lema o raíz)", "Conviene que la consulta y los documentos se normalicen igual."),
    ("Clasificación con TF-IDF (clase 5)", "opcional (TF-IDF ya pondera)", "prueba y compara", "Se decide con validación, no a ojo."),
    ("Sentimiento con negaciones (clase 6)", "conservar las negaciones", "con cuidado", "`no bueno` no es `bueno`."),
    ("Reconocimiento de entidades (clase 7)", "no", "no", "Necesita mayúsculas y contexto."),
    ("Generación de texto (clase 8)", "no", "no", "El texto generado debe ser texto completo."),
], columns=["tarea", "stopwords", "stemming / lematización", "por qué"])
print(guia.to_string(index=False))

> **Regla práctica:** aplica **exactamente el mismo** preprocesamiento en el entrenamiento y en la prueba (y en producción). Si lo cambias entre los dos, el modelo ve palabras que no conoce. Cómo evitar la fuga de información al construirlo dentro de una `Pipeline`, en la clase 5.

---
## Resumen y siguiente paso

- Las **stopwords** de NLTK incluyen algunas negaciones y no otras; quitarlas cambia el significado de frases como `no está bien` / `non è buono`. Usa una **lista propia**: conserva las negaciones y añade palabras de dominio si hace falta.
- El **stemming** (Snowball) recorta por reglas: rápido, sin garantía de palabra real, con sobre-stemming y sub-stemming.
- La **lematización** (spaCy) da formas de diccionario y depende del POS; los modelos pequeños fallan en clíticos, formas aisladas, nombres propios y texto ruidoso.
- El efecto en el **vocabulario** se mide, no se supone: minúsculas y stemming reducen mucho; quitar stopwords casi no reduce tipos.
- Hay que **aplicar lo mismo en entrenamiento y prueba**.

Continúa con la clase 4: Representaciones dispersas (bolsa de palabras y TF-IDF).